# TabPFN Local Interpretability — Native Shapiq

Fully local interpretability for **TabPFN_10K with raw features** using PriorLabs'
native `tabpfn_extensions` interpretability module.

**Why native Shapiq over KernelSHAP:**
- TabPFN's explainer removes features by re-contextualising the Bayesian in-context learning,
  which is *exactly* how TabPFN handles missing data. This makes the explanations faithful
  to the model's internal reasoning, not a generic black-box approximation.
- Typically 10–50× faster than KernelSHAP for the same budget.
- Fully local — no API calls, GPU compute only.

**Setup:**
- `exposure_strategy='rate'`: target = pure_premium / Exposure, no exposure column
  → clean 9-feature input space for interpretability
- 10K subsample (same as TabPFN_10K experiments in `post2_tabpfn.ipynb`)

**Experiments:**
1. Fit `TabPFNRegressor` with raw features + rate strategy
2. Shapiq waterfall plots — 5 key policies
3. Global feature importance — mean |SHAP| over 100 holdout rows
4. Partial dependence plots — BonusMalus, DrivAge, VehPower, Density
5. Feature selection — minimal subset preserving performance

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Auth must be set before importing tabpfn
# Set TABPFN_TOKEN in env before launching Jupyter, or set it here:
# os.environ['TABPFN_TOKEN'] = 'your_token_here'
os.environ.setdefault('TABPFN_NO_BROWSER', 'true')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Interpretability on 10K rows will be slow.')

token = os.environ.get('TABPFN_TOKEN', '')
print(f"\nTABPFN_TOKEN set: {bool(token)} (prefix: {token[:8]}...)" if token else "\nWARNING: TABPFN_TOKEN not set")

from src.data.load_insurance import load_processed, get_dev_holdout
from src.features.insurance_features import RawFeaturePipeline
from src.models.tabpfn_wrapper import TabPFNWrapper

print('Base imports OK')

In [ ]:
# Check / install tabpfn-extensions[interpretability]
# Docs: https://docs.priorlabs.ai/capabilities/interpretability
try:
    from tabpfn_extensions.interpretability.shapiq import get_tabpfn_explainer
    from tabpfn_extensions.interpretability.pdp import partial_dependence_plots
    from tabpfn_extensions.interpretability.feature_selection import feature_selection
    print('tabpfn-extensions[interpretability] already installed')
except ImportError:
    import subprocess
    print('Installing tabpfn-extensions[interpretability]...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'tabpfn-extensions[interpretability]', '-q'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('pip stderr:', result.stderr[-500:])
        raise RuntimeError('Install failed')
    from tabpfn_extensions.interpretability.shapiq import get_tabpfn_explainer
    from tabpfn_extensions.interpretability.pdp import partial_dependence_plots
    from tabpfn_extensions.interpretability.feature_selection import feature_selection
    print('Installed and imported OK')

import tabpfn_extensions
print(f'tabpfn_extensions version: {getattr(tabpfn_extensions, "__version__", "unknown")}')

## Step 1 — Data Preparation

Raw features only: `Area, VehBrand, VehGas, Region` (ordinal encoded) +
`VehPower, VehAge, DrivAge, BonusMalus, Density` (numeric) = **9 features**.

Rate strategy: `target = pure_premium / Exposure` — no exposure column in the feature
matrix. This gives a clean 9-feature space where SHAP values are directly comparable
across policies regardless of exposure duration.

In [ ]:
df = load_processed()
splits = get_dev_holdout(df)

X_dev, X_holdout     = splits['X_dev'], splits['X_holdout']
y_dev, y_holdout     = splits['y_dev'], splits['y_holdout']
exp_dev, exp_holdout = splits['exposure_dev'], splits['exposure_holdout']

# Fit RawFeaturePipeline on dev set
pipe = RawFeaturePipeline()
X_arr         = pipe.fit_transform(X_dev)       # (N_dev, 9)
X_holdout_arr = pipe.transform(X_holdout)       # (N_holdout, 9)

feat_names = pipe.feature_names_
print(f'Features ({len(feat_names)}): {feat_names}')

# Rate strategy: target = pure_premium / exposure
# Exposure is not a feature — model predicts claim rate, multiply back at inference
y_pp   = y_dev['pure_premium'].values
exp    = exp_dev.values
y_rate = y_pp / np.clip(exp, 1e-10, None)

y_pp_h   = y_holdout['pure_premium'].values
exp_h    = exp_holdout.values
y_rate_h = y_pp_h / np.clip(exp_h, 1e-10, None)

MEAN_EXP = float(np.mean(exp))  # used as fixed exposure for SHAP explanations

print(f'Dev:     {X_arr.shape[0]:,} rows')
print(f'Holdout: {X_holdout_arr.shape[0]:,} rows')
print(f'Rate target — p50: {np.median(y_rate):.4f}, p95: {np.percentile(y_rate, 95):.2f}')
print(f'Mean exposure: {MEAN_EXP:.4f}')

In [ ]:
# Subsample to 10K using same stratified logic as TabPFNWrapper
N_TRAIN = 10_000
_helper = TabPFNWrapper(task='regression', n_train_max=N_TRAIN,
                         exposure_strategy='rate', random_state=42)
X_train, y_train, _ = _helper._subsample(X_arr, y_rate)
print(f'Training data: {X_train.shape}  (subsampled from {X_arr.shape[0]:,})')

# Fit TabPFNRegressor directly — we need the bare model object for the explainer
from tabpfn import TabPFNRegressor
device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time()
model = TabPFNRegressor(device=device, random_state=42)
model.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'\nTabPFNRegressor fit in {elapsed:.1f}s on {device}')

# Smoke test: predict rates on a small holdout slice, then convert to premiums
rates_check = model.predict(X_holdout_arr[:500])
premiums_check = rates_check * exp_h[:500]
print(f'Rate predictions:    min={rates_check.min():.4f},  mean={rates_check.mean():.4f},  max={rates_check.max():.2f}')
print(f'Premium predictions: min={premiums_check.min():.4f}, mean={premiums_check.mean():.4f}, max={premiums_check.max():.2f}')

## Step 2 — Native Shapiq Explainer

The `get_tabpfn_explainer` creates a **TabPFNExplainer** backed by the `shapiq` library.

Key difference from KernelSHAP:
- **KernelSHAP** marginalises out features by replacing them with random samples from
  the background dataset — the model then predicts on hybrid rows it was never trained on.
- **TabPFN Shapiq** marginalises by *removing features from the training context*,
  then re-running inference. This matches TabPFN's Bayesian in-context mechanism
  exactly — the model never sees inputs it can't handle.

Parameters:
- `index='k-SII'`, `max_order=1` → standard Shapley values (9 values per row)
- `budget=128` per row → recommended minimum for <10 features

**Speed optimisation:** the explainer context uses a 2K subsample of the training data
(not the full 10K). Each coalition evaluation refits the model, so fewer context rows
= faster refits. TabPFN generalises well from 2K rows, so SHAP fidelity is preserved.

In [ ]:
# Minimal monkey-patch for numpy 2.0+ compatibility
# Issue: shapiq's TabPFNImputer.value_function calls float(self.predict(x))
#        but predict returns shape (1,) array — float() rejects non-0d arrays in numpy 2.0+.
# Fix:   patch predict to return a scalar when input is a single row.
#        This keeps the original optimized value_function intact.
#
# IMPORTANT: must restart kernel before running this cell if a previous
# patch was already applied, to avoid recursion on stale _orig_predict.
import shapiq.imputer.tabpfn_imputer as _tabpfn_mod
import shapiq.imputer.base as _base_mod
import numpy as _np

_OrigImputer = _tabpfn_mod.TabPFNImputer

# Get the TRUE original predict from the base Imputer class (never patched)
_base_predict = _base_mod.Imputer.predict

def _patched_predict(self, x):
    result = _base_predict(self, x)
    arr = _np.asarray(result).ravel()
    return arr.item() if arr.size == 1 else arr

_OrigImputer.predict = _patched_predict
print('Patched TabPFNImputer.predict for numpy 2.0+ compatibility')

In [ ]:
# Subsample training data for explainer — fewer context rows = faster refits per coalition
N_EXPLAINER = 500
rng_exp = np.random.default_rng(42)
exp_idx = rng_exp.choice(len(X_train), size=N_EXPLAINER, replace=False)

print(f'Creating TabPFN Shapiq explainer (context: {N_EXPLAINER} rows)...')
t0 = time.time()
explainer = get_tabpfn_explainer(
    model=model,
    data=X_train[exp_idx],
    labels=y_train[exp_idx],
    index='k-SII',
    max_order=1,   # order 1 = standard Shapley values
)
print(f'Explainer created in {time.time()-t0:.1f}s')

# Quick test on one row to verify the API
t0 = time.time()
sv_test = explainer.explain(X_holdout_arr[0:1], budget=64)
elapsed = time.time() - t0
print(f'Single-row explanation in {elapsed:.2f}s')
print(f'InteractionValues type: {type(sv_test).__name__}')
print(f'Model prediction (rate): {model.predict(X_holdout_arr[0:1])[0]:.4f}')

# Show available attributes for introspection
print(f'Attributes: {[a for a in dir(sv_test) if not a.startswith("_")]}')

In [ ]:
# Test waterfall plot on a single row to confirm the display works
sv_test.feature_names = feat_names   # attach names if supported
fig, ax = plt.subplots(figsize=(8, 4))
sv_test.plot_waterfall(ax=ax) if hasattr(sv_test, 'plot_waterfall') else plt.text(0.5, 0.5, 'plot not available', ha='center')
ax.set_title('Single row — TabPFN Shapiq waterfall test', fontsize=10)
plt.tight_layout()
plt.show()

## Step 3 — Global Feature Importance

Explain 100 holdout rows, compute **mean |SHAP|** per feature.

Expected runtime: ~0.5–2s per row on GPU → 1–3 min total.
Compare this to KernelSHAP on 200 rows which took 5–30 min.

In [ ]:
N_GLOBAL = 100
BUDGET   = 64

rng_g = np.random.default_rng(77)
explain_idx = rng_g.choice(len(X_holdout_arr), size=N_GLOBAL, replace=False)
X_explain = X_holdout_arr[explain_idx]

print(f'Computing Shapiq explanations: {N_GLOBAL} rows × budget={BUDGET}...')
t0 = time.time()

shap_matrix = np.zeros((N_GLOBAL, len(feat_names)))
errors = []

for i in range(N_GLOBAL):
    if i % 25 == 0:
        elapsed = time.time() - t0
        rate = elapsed / max(i, 1)
        eta  = rate * (N_GLOBAL - i)
        print(f'  {i}/{N_GLOBAL}  ({elapsed:.0f}s elapsed, ETA {eta:.0f}s)')
    try:
        sv = explainer.explain(X_explain[i:i+1], budget=BUDGET)
        # Extract per-feature Shapley values
        # InteractionValues with max_order=1: keys are (feature_idx,) tuples
        for j in range(len(feat_names)):
            shap_matrix[i, j] = float(sv[(j,)])
    except Exception as e:
        errors.append((i, str(e)))
        # Leave as zeros — will be visible in importance plot

total_time = time.time() - t0
print(f'\nDone: {total_time:.1f}s total ({total_time/N_GLOBAL:.2f}s/row)')
if errors:
    print(f'Errors on {len(errors)} rows: {errors[:3]}')

# Global importance: mean absolute SHAP value per feature
importance = np.abs(shap_matrix).mean(axis=0)
imp_series = pd.Series(importance, index=feat_names).sort_values(ascending=False)
print('\nFeature importance (mean |SHAP|):')
print(imp_series.round(6).to_string())

In [ ]:
fig_dir = Path(project_root) / 'figures' / 'post2'
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

order = np.argsort(importance)
norm_imp = importance / (importance.max() + 1e-12)

bars = ax.barh(
    [feat_names[i] for i in order],
    norm_imp[order],
    color='#FF6F00', alpha=0.85
)
ax.set_xlabel('Normalised mean |SHAP|  (1.0 = most important feature)', fontsize=10)
ax.set_title(
    f'TabPFN_10K Raw — Feature Importance (Shapiq, {N_GLOBAL} holdout rows)\n'
    f'Rate strategy: target = pure_premium / Exposure, 9 features',
    fontsize=10
)
ax.axvline(0, color='black', linewidth=0.5)

# Label bars with rank
for idx, (feat_i, val) in enumerate(zip(order, norm_imp[order])):
    ax.text(val + 0.01, idx, f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
fname = fig_dir / 'shapiq_global_importance_tabpfn10k_raw.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {fname}')

## Step 4 — Waterfall Plots: 5 Key Policies

Same 5-policy selection as the KernelSHAP section in `post2_tabpfn.ipynb`:
- **A** — lowest predicted rate (safest driver)
- **B** — median predicted rate
- **C** — highest predicted rate (riskiest driver)
- **D** — policy with BonusMalus closest to 100 (malus boundary)
- **E** — young driver (<26) with high VehPower (>10)

Each waterfall shows: how much each of the 9 features pushed the prediction
above or below the expected value (mean prediction over training set).

In [ ]:
# Use the 100-row explain set for policies A, B, C
# For D and E, search the full holdout for matching criteria
preds_explain = model.predict(X_explain)

policy_map = {
    'A_low_risk':    int(np.argmin(preds_explain)),
    'B_median_risk': int(np.argsort(preds_explain)[len(preds_explain)//2]),
    'C_high_risk':   int(np.argmax(preds_explain)),
}

# D: policy with BonusMalus closest to 100 in explain set
bm_col_idx = feat_names.index('BonusMalus')
bm_vals    = X_explain[:, bm_col_idx]
policy_map['D_malus_boundary'] = int(np.argmin(np.abs(bm_vals - 100.0)))

# E: young driver (DrivAge < 26) with highest VehPower in explain set
age_col_idx = feat_names.index('DrivAge')
pow_col_idx = feat_names.index('VehPower')
young_mask  = X_explain[:, age_col_idx] < 26
if young_mask.any():
    young_rows = np.where(young_mask)[0]
    policy_map['E_young_high_power'] = int(young_rows[np.argmax(X_explain[young_rows, pow_col_idx])])
else:
    # Fallback: youngest driver in explain set
    policy_map['E_young_high_power'] = int(np.argmin(X_explain[:, age_col_idx]))

print('Selected policies (index within explain set):')
for label, idx in policy_map.items():
    row = X_explain[idx]
    print(f'  {label:22s}  idx={idx:3d}  rate={preds_explain[idx]:.4f}  '
          f'BM={row[bm_col_idx]:.0f}  Age={row[age_col_idx]:.0f}  Pow={row[pow_col_idx]:.0f}')

In [ ]:
for label, row_idx in policy_map.items():
    t0 = time.time()
    sv = explainer.explain(X_explain[row_idx:row_idx+1], budget=64)
    elapsed = time.time() - t0

    pred_rate    = preds_explain[row_idx]
    pred_premium = pred_rate * MEAN_EXP  # convert to premium at mean exposure

    fig, ax = plt.subplots(figsize=(9, 4))

    # Attach feature names if the InteractionValues object supports it
    if hasattr(sv, 'feature_names'):
        sv.feature_names = feat_names

    try:
        sv.plot_waterfall(ax=ax)
    except TypeError:
        # Some versions don't accept ax as kwarg — use current axes
        plt.sca(ax)
        sv.plot_waterfall()
    except Exception as e:
        # Fallback: manual bar chart from extracted values
        shap_vals = np.array([sv[(j,)] for j in range(len(feat_names))])
        order_w   = np.argsort(np.abs(shap_vals))[::-1]
        colors    = ['#e74c3c' if v > 0 else '#3498db' for v in shap_vals[order_w]]
        ax.barh([feat_names[i] for i in order_w], shap_vals[order_w], color=colors)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel('SHAP value (rate scale)', fontsize=9)
        print(f'  Fallback bar chart used: {e}')

    policy_label = label.replace('_', ' ')
    ax.set_title(
        f'{policy_label}  |  rate={pred_rate:.4f}  →  premium ≈ {pred_premium:.2f}€ (at mean exposure)\n'
        f'BonusMalus={X_explain[row_idx, bm_col_idx]:.0f}  '
        f'DrivAge={X_explain[row_idx, age_col_idx]:.0f}  '
        f'VehPower={X_explain[row_idx, pow_col_idx]:.0f}  '
        f'(Shapiq {elapsed:.1f}s)',
        fontsize=9
    )

    plt.tight_layout()
    fname = fig_dir / f'shapiq_waterfall_{label}_tabpfn10k_raw.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {fname}')

## Step 5 — Partial Dependence Plots

PDPs show the **average marginal effect** of a single feature on the predicted claim rate,
with all other features held at their observed values.

Key features:
- **BonusMalus** (idx 7): primary pricing signal — expect strong non-linear increase
- **DrivAge** (idx 6): U-shaped actuarial risk curve — high at young and senior ages
- **VehPower** (idx 4): flattens above ~10
- **Density** (idx 8): urban/rural gradient (raw Density, log-compressed implicitly by TabPFN)

In [ ]:
# Use a fixed 500-row subsample for PDPs (speed vs coverage trade-off)
rng_pdp = np.random.default_rng(55)
pdp_idx  = rng_pdp.choice(len(X_holdout_arr), size=500, replace=False)
X_pdp    = X_holdout_arr[pdp_idx]

KEY_FEATURES = [
    (feat_names.index('BonusMalus'), 'BonusMalus',  'BonusMalus (no-claims discount score)'),
    (feat_names.index('DrivAge'),    'DrivAge',      'DrivAge (years)'),
    (feat_names.index('VehPower'),   'VehPower',     'VehPower (fiscal horsepower)'),
    (feat_names.index('Density'),    'Density',      'Density (population per km²)'),
]

print('Generating Partial Dependence Plots...')
print('(tabpfn_extensions PDP uses ICE curves internally — expect ~30s per feature)')

for feat_idx, feat_key, feat_label in KEY_FEATURES:
    t0 = time.time()
    try:
        fig = partial_dependence_plots(
            model,
            X_pdp,
            features=[feat_idx],
            kind='average',
        )
        if fig is not None:
            ax = fig.axes[0] if hasattr(fig, 'axes') else plt.gca()
            ax.set_xlabel(feat_label, fontsize=11)
            ax.set_ylabel('Average predicted claim rate', fontsize=11)
            ax.set_title(f'PDP — {feat_label} | TabPFN_10K Raw', fontsize=10)
            plt.tight_layout()
            fname = fig_dir / f'shapiq_pdp_{feat_key}_tabpfn10k_raw.png'
            plt.savefig(fname, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'  {feat_label}: {time.time()-t0:.1f}s → {fname}')
        else:
            print(f'  {feat_label}: partial_dependence_plots returned None — checking API')
    except Exception as e:
        print(f'  ERROR on {feat_label}: {e}')
        # Fallback: manual PDP using model.predict
        X_sweep = X_pdp.copy()
        feat_values = np.percentile(X_holdout_arr[:, feat_idx], np.linspace(5, 95, 30))
        mean_preds  = []
        for val in feat_values:
            X_sw = X_sweep.copy()
            X_sw[:, feat_idx] = val
            mean_preds.append(model.predict(X_sw).mean())
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(feat_values, mean_preds, 'o-', color='#FF6F00', linewidth=2, markersize=4)
        ax.set_xlabel(feat_label, fontsize=11)
        ax.set_ylabel('Mean predicted rate', fontsize=11)
        ax.set_title(f'PDP (manual) — {feat_label} | TabPFN_10K Raw', fontsize=10)
        plt.tight_layout()
        fname = fig_dir / f'shapiq_pdp_{feat_key}_tabpfn10k_raw.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Fallback PDP saved → {fname} ({time.time()-t0:.1f}s)')

## Step 6 — Feature Selection

Identifies the **minimal feature subset** that preserves predictive performance.

This is particularly interesting for TabPFN with raw features: do we need all 9,
or does BonusMalus + DrivAge alone explain most of the variance?

Method: iterative backward elimination guided by TabPFN's context-window scoring.
`n_features_to_select=5` removes the 4 least informative features.

In [ ]:
print('Running feature selection (n_features_to_select=5)...')
t0 = time.time()

try:
    selector = feature_selection(
        model,
        X_train,
        y_train,
        n_features_to_select=5,
    )
    elapsed = time.time() - t0
    print(f'Feature selection done in {elapsed:.1f}s')

    # Selected features
    selected_mask = selector.get_support()
    selected_feat = [feat_names[i] for i, s in enumerate(selected_mask) if s]
    removed_feat  = [feat_names[i] for i, s in enumerate(selected_mask) if not s]
    print(f'Selected ({len(selected_feat)}): {selected_feat}')
    print(f'Removed  ({len(removed_feat)}):  {removed_feat}')

    # Evaluate: fit new 10K model on selected features and compare holdout MSE
    X_train_sel   = selector.transform(X_train)
    X_holdout_sel = selector.transform(X_holdout_arr)

    model_sel = TabPFNRegressor(random_state=42)
    model_sel.fit(X_train_sel, y_train)

    preds_full = model.predict(X_holdout_arr)
    preds_sel  = model_sel.predict(X_holdout_sel)

    # Compare rate-scale MAE
    mae_full = np.mean(np.abs(preds_full - y_rate_h))
    mae_sel  = np.mean(np.abs(preds_sel  - y_rate_h))
    print(f'\nHoldout MAE (rate scale):')
    print(f'  All 9 features: {mae_full:.6f}')
    print(f'  Top 5 features: {mae_sel:.6f}  (delta: {mae_sel - mae_full:+.6f})')

    # Correlation between full and selected predictions
    corr = np.corrcoef(preds_full, preds_sel)[0, 1]
    print(f'  Pearson r (full vs selected predictions): {corr:.4f}')

except Exception as e:
    print(f'Feature selection API error: {e}')
    print('Falling back to importance-based selection using Shapiq results...')

    # Rank by global Shapiq importance computed in Step 3
    top5_idx     = np.argsort(importance)[::-1][:5]
    selected_feat = [feat_names[i] for i in top5_idx]
    removed_feat  = [f for f in feat_names if f not in selected_feat]
    print(f'Top 5 by Shapiq importance: {selected_feat}')
    print(f'Removed: {removed_feat}')

## Step 7 — Summary

Comparison of native Shapiq vs KernelSHAP (from `post2_tabpfn.ipynb`):

In [ ]:
print('=== TabPFN_10K Raw — Shapiq Interpretability Summary ===')
print(f'Features:          {len(feat_names)} raw features (rate strategy, no exposure column)')
print(f'Training rows:     {N_TRAIN:,}')
print(f'Explained rows:    {N_GLOBAL}')
print(f'Budget per row:    {BUDGET}')
print(f'Total SHAP time:   {total_time:.1f}s  ({total_time/N_GLOBAL:.2f}s/row)')
print()
print('Feature importance ranking (Shapiq, mean |SHAP|):')
for rank, (feat, imp) in enumerate(imp_series.items(), 1):
    bar = '█' * int(imp / imp_series.max() * 20)
    print(f'  {rank:2d}. {feat:15s}  {imp:.6f}  {bar}')
print()
print('Artifacts saved to figures/post2/:')
print('  shapiq_global_importance_tabpfn10k_raw.png')
print('  shapiq_waterfall_{label}_tabpfn10k_raw.png  (×5)')
print('  shapiq_pdp_{feature}_tabpfn10k_raw.png       (×4)')